In [6]:
import os
from deepeval import assert_test
from deepeval.test_case import LLMTestCase
from deepeval.metrics import AnswerRelevancyMetric

os.environ["OPENAI_API_KEY"] = "sk-4408eca08dd7492890a48bacebe294c8"
os.environ["OPENAI_MODEL_NAME"] = "qwen3.6-flash-2026-04-16"
os.environ["OPENAI_BASE_URL"] = "https://dashscope.aliyuncs.com/compatible-mode/v1"

## 示例 1：基础概念——什么是 TestCase

In [7]:
# 一个测试用例 = 一次 RAG 问答的完整记录
test_case = LLMTestCase(
    input="故意杀人罪怎么判？",              # 用户提问
    actual_output="根据《刑法》第232条，故意杀人的，处死刑、无期徒刑或者十年以上有期徒刑。",  # RAG系统生成的回答，不用手动写
    expected_output="故意杀人的，处死刑、无期徒刑或者十年以上有期徒刑。情节较轻的，处三年以上十年以下有期徒刑。",  # 标准答案（人工标注）
    retrieval_context=[                      # RAG 检索到的上下文片段（从向量库召回的原文），不用手动写
        "第二百三十二条 故意杀人的，处死刑、无期徒刑或者十年以上有期徒刑。情节较轻的，处三年以上十年以下有期徒刑。",
        "第二百三十三条 过失致人死亡的，处三年以上七年以下有期徒刑；情节较轻的，处三年以下有期徒刑。",
    ],
)

# 参数详解：
# input：        用户输入的问题，评估"回答相关性"时用
# actual_output： 你的 RAG 系统实际给出的回答，所有指标都会评估它
# expected_output： 人工标注的标准答案，评估"准确性"时用（部分指标不需要）
# retrieval_context： 检索到的上下文片段列表，评估"忠实性""上下文精度"时用

metric = AnswerRelevancyMetric()
assert_test(test_case, [metric])


## 示例 2：Answer Relevancy——回答相关性

In [8]:
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase

# 创建指标实例
answer_relevancy = AnswerRelevancyMetric(
    threshold=0.5,          # 及格线：分数 >= 0.5 算通过，低于 0.5 算不通过
                            # 范围 0~1，建议设 0.5，太严格容易误判
    model="qwen3.6-flash-2026-04-16",  # 用来打分的 LLM，需要支持 function calling
                            # 你用 DeepSeek 的话填 "deepseek-chat"
    include_reason=True,    # 返回结果中是否包含"为什么给这个分"的解释
                            # True = 返回分数 + 文字说明，方便调试
)

test_case = LLMTestCase(
    input="刑法中故意杀人罪怎么处罚？",
    actual_output="根据《刑法》第232条，故意杀人的，处死刑、无期徒刑或者十年以上有期徒刑。",
)

# 执行评估
answer_relevancy.measure(test_case)

print(f"分数: {answer_relevancy.score}")          # 0~1 之间的浮点数
print(f"是否通过: {answer_relevancy.is_successful()}")  # True/False
print(f"原因: {answer_relevancy.reason}")          # 为什么给这个分的文字解释

# score 接近 1 → 回答高度切题
# score 接近 0 → 答非所问，比如问了杀人罪却回答了盗窃罪
# 这个指标只需要 input 和 actual_output，不需要 retrieval_context


分数: 1.0
是否通过: True
原因: The score is 1.00 because the response directly and comprehensively answers the question regarding the punishment for intentional homicide under criminal law, with zero irrelevant content detracting from its relevance. Excellent focus and precision!


## 示例 3：Faithfulness——忠实性（幻觉检测）​

In [9]:
from deepeval.metrics import FaithfulnessMetric

faithfulness = FaithfulnessMetric(
    threshold=0.7,          # 及格线，忠实性建议设高一点，0.7 比较合理
    model="qwen3.6-flash-2026-04-16",
    include_reason=True,
)

test_case = LLMTestCase(
    input="抢劫罪的最高刑期是多少？",
    actual_output="抢劫罪最高可判处死刑，特别是入户抢劫的情况。",  # 系统回答
    retrieval_context=[     # 检索到的上下文
        "第二百六十三条 以暴力、胁迫或者其他方法抢劫公私财物的，处三年以上十年以下有期徒刑，并处罚金。",
        "有下列情形之一的，处十年以上有期徒刑、无期徒刑或者死刑，并处罚金或者没收财产：（一）入户抢劫的；",
    ],
)

faithfulness.measure(test_case)

print(f"分数: {faithfulness.score}")
print(f"是否通过: {faithfulness.is_successful()}")
print(f"原因: {faithfulness.reason}")


# score = 1.0 → 回答中的每个断言都能在 retrieval_context 中找到依据
# score < 1.0 → 回答中有上下文不支持的内容（幻觉）
#
# 上面这个例子，"抢劫罪最高可判处死刑"在上下文中有依据（"处十年以上有期徒刑、无期徒刑或者死刑"），所以忠实性分数应该很高
#
# 如果 actual_output 写成"抢劫罪一律判处死刑"，这就是幻觉，
# 因为上下文说的是"有下列情形之一"才可能判死刑，不是"一律"
#
# 这个指标需要 actual_output + retrieval_context


/opt/miniconda3/envs/pytorch/lib/python3.11/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for 
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

分数: 1.0
是否通过: True
原因: The score is 1.00 because the actual output aligns perfectly with the retrieval context, demonstrating complete faithfulness with zero contradictions. Excellent work!


## 示例 4：Contextual Precision——上下文精度

In [10]:
from deepeval.metrics import ContextualPrecisionMetric

contextual_precision = ContextualPrecisionMetric(
    threshold=0.5,          # 及格线
    model="qwen3.6-flash-2026-04-16",
    include_reason=True,
)

test_case = LLMTestCase(
    input="刑法中故意杀人罪怎么处罚？",
    expected_output="故意杀人的，处死刑、无期徒刑或者十年以上有期徒刑。",
    retrieval_context=[
        # 第1条：直接相关（故意杀人罪），排在前面 → 好
        "第二百三十二条 故意杀人的，处死刑、无期徒刑或者十年以上有期徒刑。",
        # 第2条：不太相关（过失致人死亡），排在后面 → 还行
        "第二百三十三条 过失致人死亡的，处三年以上七年以下有期徒刑。",
        # 第3条：完全不相关（合同法），排在最后 → 不影响太多
        "第四百二十四条 当事人一方不履行合同义务或者履行合同义务不符合约定的，应当承担继续履行、采取补救措施或者赔偿损失等违约责任。",
    ],
)

contextual_precision.measure(test_case)

print(f"分数: {contextual_precision.score}")
print(f"原因: {contextual_precision.reason}")


# 相关内容排得越靠前，分数越高
# 如果检索结果中合同法排在第1条、刑法第232条排在第3条 → 分数会很低
# 这说明你的检索排序有问题，需要优化 embedding 或加 reranker
#
# 这个指标需要 expected_output + retrieval_context
# expected_output 用来判断每条 retrieval_context 是否与问题相关


分数: 1.0
原因: The score is 1.00 because it has already reached the maximum possible value, reflecting a flawless ranking where relevant nodes in retrieval contexts correctly precede irrelevant ones. The 1st node in retrieval contexts directly answers the question by explicitly stating the penalty for intentional homicide: '故意杀人的，处死刑、无期徒刑或者十年以上有期徒刑。' Meanwhile, the 2nd node in retrieval contexts refers to '过失致人死亡的' (negligent causing of death), which is a different offense and does not provide any information on how intentional homicide is punished, and the 3rd node in retrieval contexts discusses civil contract liability for '不履行合同义务', which is completely unrelated to criminal law or the punishment for intentional homicide. This precise ordering ensures irrelevant nodes in retrieval contexts are properly ranked below the relevant one, yielding an excellent result!


## 示例 5：Contextual Recall——上下文召回率

In [12]:
from deepeval.metrics import ContextualRecallMetric

contextual_recall = ContextualRecallMetric(
    threshold=0.7,
    model="qwen3.6-flash-2026-04-16",
    include_reason=True,
)

test_case = LLMTestCase(
    input="刑法中故意杀人罪怎么处罚？",
    expected_output="故意杀人的，处死刑、无期徒刑或者十年以上有期徒刑。情节较轻的，处三年以上十年以下有期徒刑。",
    retrieval_context=[
        # 只检索到了前半段，缺少"情节较轻"的部分
        "第二百三十二条 故意杀人的，处死刑、无期徒刑或者十年以上有期徒刑。",
    ],
)

contextual_recall.measure(test_case)

print(f"分数: {contextual_recall.score}")
print(f"原因: {contextual_recall.reason}")

# 解读：
# 标准答案有2个关键信息点，但检索上下文只覆盖了1个 → 召回率约 0.5
# 如果检索到的上下文包含了完整的第232条（含"情节较轻"的补充规定）→ 召回率会接近 1.0
# 召回率低说明检索不够全，可能需要调大 top-k 或优化分块策略
#
# 这个指标需要 expected_output + retrieval_context


分数: 0.5
原因: The score is 0.50 because sentence 1 of the expected output is directly supported by node 1 in retrieval context, whereas sentence 2 cannot be attributed to any node in retrieval context due to the omission of provisions for lighter circumstances.


## 示例 6：批量评估——组合所有指标，一次跑完

In [13]:
from deepeval.test_case import LLMTestCase
from deepeval.metrics import (
    AnswerRelevancyMetric,
    FaithfulnessMetric,
    ContextualPrecisionMetric,
    ContextualRecallMetric,
)
from deepeval import evaluate

# 1. 初始化所有指标
metrics = [
    AnswerRelevancyMetric(threshold=0.5, model="qwen3.6-flash-2026-04-16", include_reason=True),
    FaithfulnessMetric(threshold=0.7, model="qwen3.6-flash-2026-04-16", include_reason=True),
    ContextualPrecisionMetric(threshold=0.5, model="qwen3.6-flash-2026-04-16", include_reason=True),
    ContextualRecallMetric(threshold=0.7, model="qwen3.6-flash-2026-04-16", include_reason=True),
]

# 2. 构建多个测试用例（模拟你的法律 RAG 的多次问答）
test_cases = [
    LLMTestCase(
        input="故意杀人罪怎么判？",
        actual_output="根据《刑法》第232条，故意杀人的，处死刑、无期徒刑或者十年以上有期徒刑。",
        expected_output="故意杀人的，处死刑、无期徒刑或者十年以上有期徒刑。情节较轻的，处三年以上十年以下有期徒刑。",
        retrieval_context=[
            "第二百三十二条 故意杀人的，处死刑、无期徒刑或者十年以上有期徒刑。情节较轻的，处三年以上十年以下有期徒刑。",
        ],
    ),
    LLMTestCase(
        input="劳动合同试用期最长多久？",
        actual_output="根据《劳动法》规定，劳动合同试用期最长不超过六个月。",
        expected_output="劳动合同期限三个月以上不满一年的，试用期不得超过一个月；一年以上不满三年的，试用期不得超过二个月；三年以上固定期限和无固定期限的劳动合同，试用期不得超过六个月。",
        retrieval_context=[
            "第十九条 劳动合同期限三个月以上不满一年的，试用期不得超过一个月；一年以上不满三年的，试用期不得超过二个月；三年以上固定期限和无固定期限的劳动合同，试用期不得超过六个月。",
        ],
    ),
]

# 3. 一键批量评估
results = evaluate(test_cases=test_cases, metrics=metrics)

# 输出结果会显示：
# - 每个测试用例在每项指标上的分数
# - 是否通过（>= threshold）
# - 原因说明


✨ You're running DeepEval's latest Answer Relevancy Metric! (using qwen3.6-flash-2026-04-16, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Faithfulness Metric! (using qwen3.6-flash-2026-04-16, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Contextual Precision Metric! (using qwen3.6-flash-2026-04-16, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Contextual Recall Metric! (using qwen3.6-flash-2026-04-16, strict=False, 
async_mode=True)...



Metrics Summary

  - ✅ Answer Relevancy (score: 1.0, threshold: 0.5, strict: False, evaluation model: qwen3.6-flash-2026-04-16, reason: The score is 1.00 because the response directly and accurately addresses the sentencing standards for intentional homicide without any irrelevant or off-topic content, making it perfectly aligned with the query. Excellent work!, error: None)
  - ✅ Faithfulness (score: 1.0, threshold: 0.7, strict: False, evaluation model: qwen3.6-flash-2026-04-16, reason: The score is 1.00 because there are no contradictions present, meaning the actual output perfectly aligns with the retrieval context and faithfully represents all the provided information. Excellent work!, error: None)
  - ✅ Contextual Precision (score: 1.0, threshold: 0.5, strict: False, evaluation model: qwen3.6-flash-2026-04-16, reason: The score is 1.00 because the first node in the retrieval contexts perfectly addresses the query by stating "故意杀人的，处死刑、无期徒刑或者十年以上有期徒刑。情节较轻的，处三年以上十年以下有期徒刑。", which 

⚠ WARNING: No hyperparameters logged.
» ]8;id=858917;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 41.97s | token cost: None)
» Test Results (2 total tests):
   » Pass Rate: 100.0% | Passed: 2 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

## 示例 7：结合你的法律 RAG 系统做端到端评估

In [14]:
"""
端到端评估：用 DeepEval 评估你的法律 RAG 系统的实际效果
运行前确保：
1. 向量库已经构建好
2. 已设置 OPENAI_API_KEY 和 OPENAI_API_BASE
"""

from deepeval.test_case import LLMTestCase
from deepeval.metrics import (
    AnswerRelevancyMetric,
    FaithfulnessMetric,
    ContextualRecallMetric,
    ContextualPrecisionMetric,
)
from deepeval import evaluate

# 导入你项目中的模块
from rag_pipeline import load_vector_store, create_qa_chain


def evaluate_rag(test_questions: list[dict]):
    """
    端到端评估 RAG 系统

    test_questions 格式：
    [
        {
            "question": "故意杀人罪怎么判？",
            "expected_output": "故意杀人的，处死刑...",  # 人工标注的标准答案
        },
        ...
    ]
    """

    # 加载你的向量库和问答链
    vectorstore = load_vector_store()
    qa_chain, retriever = create_qa_chain(vectorstore, k=3)

    # 初始化评估指标
    metrics = [
        AnswerRelevancyMetric(threshold=0.5, model="qwen3.6-flash-2026-04-16", include_reason=True),
        FaithfulnessMetric(threshold=0.7, model="qwen3.6-flash-2026-04-16", include_reason=True),
        ContextualRecallMetric(threshold=0.7, model="qwen3.6-flash-2026-04-16", include_reason=True),
        ContextualPrecisionMetric(threshold=0.5, model="qwen3.6-flash-2026-04-16", include_reason=True),
    ]

    test_cases = []

    for item in test_questions:
        question = item["question"]
        expected = item["expected_output"]

        # 1. 用你的 RAG 系统检索上下文
        retrieved_docs = retriever.invoke(question)
        retrieval_context = [doc.page_content for doc in retrieved_docs]

        # 2. 用你的 RAG 系统生成回答
        answer = qa_chain.invoke(question)

        # 3. 构建测试用例
        test_case = LLMTestCase(
            input=question,
            actual_output=answer,
            expected_output=expected,
            retrieval_context=retrieval_context,
        )
        test_cases.append(test_case)

        print(f"✅ 已收集: {question}")
        print(f"   回答: {answer[:80]}...")
        print(f"   检索到 {len(retrieved_docs)} 条上下文")

    # 4. 批量评估
    results = evaluate(test_cases=test_cases, metrics=metrics)
    return results


# ===== 运行评估 =====
if __name__ == "__main__":
    # 准备测试问题（人工编写 + 标注标准答案）
    test_questions = [
        {
            "question": "故意杀人罪的刑罚是什么？",
            "expected_output": "故意杀人的，处死刑、无期徒刑或者十年以上有期徒刑。情节较轻的，处三年以上十年以下有期徒刑。",
        },
        {
            "question": "劳动合同试用期有哪些规定？",
            "expected_output": "劳动合同期限三个月以上不满一年的，试用期不得超过一个月；一年以上不满三年的，试用期不得超过二个月；三年以上固定期限和无固定期限的劳动合同，试用期不得超过六个月。同一用人单位与同一劳动者只能约定一次试用期。",
        },
        {
            "question": "什么情形下属于抢劫罪的加重情节？",
            "expected_output": "入户抢劫的；在公共交通工具上抢劫的；抢劫银行或者其他金融机构的；多次抢劫或者抢劫数额巨大的；抢劫致人重伤、死亡的；冒充军警人员抢劫的；持枪抢劫的；抢劫军用物资或者抢险、救灾、救济物资的。",
        },
    ]

    evaluate_rag(test_questions)


/opt/miniconda3/envs/pytorch/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6299.27it/s]
BertModel LOAD REPORT from: shibing624/text2vec-base-chinese
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ 已收集: 故意杀人罪的刑罚是什么？
   回答: 根据检索到的法律法规上下文，关于故意杀人罪的刑罚规定如下：

**《刑法》第二百三十二条**明确规定：“故意杀人的，处死刑、无期徒刑或者十年以上有期徒刑；情节较...
   检索到 3 条上下文
✅ 已收集: 劳动合同试用期有哪些规定？
   回答: 根据您提供的检索资料，关于劳动合同试用期的规定主要集中在《中华人民共和国劳动法》中，具体如下：

1. **试用期的约定与最长期限**：《中华人民共和国劳动法》...
   检索到 3 条上下文
✅ 已收集: 什么情形下属于抢劫罪的加重情节？
   回答: 根据您提供的检索资料，关于抢劫罪的加重情节，明确规定于**《中华人民共和国刑法》第二百六十三条**。

该条文指出，以暴力、胁迫或者其他方法抢劫公私财物的，基础...
   检索到 3 条上下文


✨ You're running DeepEval's latest Answer Relevancy Metric! (using qwen3.6-flash-2026-04-16, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Faithfulness Metric! (using qwen3.6-flash-2026-04-16, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Contextual Recall Metric! (using qwen3.6-flash-2026-04-16, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Contextual Precision Metric! (using qwen3.6-flash-2026-04-16, strict=False, 
async_mode=True)...

/opt/miniconda3/envs/pytorch/lib/python3.11/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for 
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')



Metrics Summary

  - ✅ Answer Relevancy (score: 0.9, threshold: 0.5, strict: False, evaluation model: qwen3.6-flash-2026-04-16, reason: 该回答主体准确说明了故意杀人罪的法定刑罚，因此保持了较高的相关性；但其中包含了一段关于如何寻求法律帮助的实务建议，与问题核心完全无关，导致得分未能达到满分，故评定为0.90。, error: None)
  - ✅ Faithfulness (score: 1.0, threshold: 0.7, strict: False, evaluation model: qwen3.6-flash-2026-04-16, reason: The score is 1.00 because the actual output is completely faithful to the retrieval context, with the contradiction list entirely empty. Excellent work!, error: None)
  - ✅ Contextual Recall (score: 1.0, threshold: 0.7, strict: False, evaluation model: qwen3.6-flash-2026-04-16, reason: The score is 1.00 because Sentence 1 and Sentence 2 in the expected output are perfectly aligned with node 1 in retrieval context, showcasing complete coverage and outstanding contextual recall., error: None)
  - ✅ Contextual Precision (score: 1.0, threshold: 0.5, strict: False, evaluation model: qwen3.6-flash-2026-04-16, reason: The score is 1.00 because 

⚠ WARNING: No hyperparameters logged.
» ]8;id=702001;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 161.71s | token cost: None)
» Test Results (3 total tests):
   » Pass Rate: 66.67% | Passed: 2 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.